# Tratamento da Base de Execução Orçamentária 2024

Notebook para baixar/ler o arquivo `basedadosexecucao_1224.csv`, validar estrutura mínima,
padronizar tipos das colunas de valor (`Vl_*`) e exportar um arquivo final com colunas fixas
para consumo no pipeline do DataSP.

In [ ]:
import pandas as pd
from datetime import datetime
from os import environ, makedirs, path

In [ ]:
DEFAULT_SOURCE_URL = 'https://orcamento.sf.prefeitura.sp.gov.br/orcamento/uploads/2024/basedadosexecucao_1224.csv'
DEFAULT_FALLBACK_SOURCE_URL = 'https://orcamento.sf.prefeitura.sp.gov.br/orcamento/uploads/2025/basedadosexecucaoConsolidadosComRestos_1225.csv'

SOURCE_URL = environ.get('ORCAMENTO_EXECUCAO_URL', DEFAULT_SOURCE_URL)
FALLBACK_SOURCE_URL = environ.get('ORCAMENTO_EXECUCAO_FALLBACK_URL', DEFAULT_FALLBACK_SOURCE_URL)
TARGET_YEAR = str(environ.get('ORCAMENTO_EXECUCAO_ANO', '2024'))

OUTPUT_DIR = path.join('data_output', 'urbanismo')
OUTPUT_FILENAME = 'orcamento_2024.csv'
OUTPUT_PATH = path.join(OUTPUT_DIR, OUTPUT_FILENAME)

EXPECTED_COLUMNS = ['Cd_AnoExecucao', 'Cd_Exercicio', 'Cd_Dotacao_Id', 'Administracao', 'Cd_Orgao', 'Sigla_Orgao', 'Ds_Orgao', 'Cd_Funcao', 'Ds_Funcao', 'Cd_SubFuncao', 'Ds_SubFuncao', 'Cd_Programa', 'Ds_Programa', 'ProjetoAtividade', 'Ds_Projeto_Atividade', 'Vl_Orcado_Ano', 'Vl_Suplementado', 'Vl_Reduzido', 'Vl_SuplementadoLiquido', 'Vl_Orcado_Atualizado', 'Vl_ReservadoLiquido', 'Vl_EmpenhadoLiquido', 'Vl_Liquidado', 'Vl_Pago']
VALUE_COLUMNS = [col for col in EXPECTED_COLUMNS if col.startswith('Vl_')]
PROCESSING_DATE = datetime.now().strftime('%Y-%m-%d')

print(f'Fonte principal: {SOURCE_URL}')
print(f'Fonte fallback: {FALLBACK_SOURCE_URL}')
print(f'Ano alvo: {TARGET_YEAR}')
print(f'Data de processamento: {PROCESSING_DATE}')

In [ ]:
df_raw = pd.read_csv(SOURCE_URL, sep=';', encoding='latin1', dtype=str)
rows_before = len(df_raw)
print(f'Linhas na origem: {rows_before:,}')
print(f'Colunas na origem: {len(df_raw.columns)}')
df_raw.head()

In [ ]:
missing_columns = [col for col in EXPECTED_COLUMNS if col not in df_raw.columns]
used_source_url = SOURCE_URL
source_mode = 'fonte_principal'

if missing_columns:
    preview_columns = df_raw.columns[:20].tolist()
    print('Layout inesperado na fonte principal. Tentando fallback consolidado com restos...')
    print(f'Colunas faltantes na fonte principal: {missing_columns}')
    print(f'Primeiras colunas encontradas: {preview_columns}')

    df_raw_fallback = pd.read_csv(FALLBACK_SOURCE_URL, sep=';', encoding='latin1', dtype=str)
    missing_fallback = [col for col in EXPECTED_COLUMNS if col not in df_raw_fallback.columns]

    if missing_fallback:
        preview_fallback = df_raw_fallback.columns[:20].tolist()
        raise ValueError(
            'Falha de layout na fonte principal e no fallback. '
            f'Faltantes na principal: {missing_columns}. '
            f'Faltantes no fallback: {missing_fallback}. '
            f'Primeiras colunas do fallback: {preview_fallback}'
        )

    rows_fallback_before_filter = len(df_raw_fallback)
    df_raw = df_raw_fallback.loc[
        (df_raw_fallback['Cd_Exercicio'].astype(str) == TARGET_YEAR)
        & (df_raw_fallback['Cd_AnoExecucao'].astype(str) == TARGET_YEAR)
    ].copy()

    rows_before = len(df_raw)
    used_source_url = FALLBACK_SOURCE_URL
    source_mode = 'fallback_consolidado_com_restos'

    print(f'Fallback aplicado com sucesso. Linhas antes do filtro: {rows_fallback_before_filter:,}')
    print(f'Linhas ap?s filtro Cd_Exercicio == {TARGET_YEAR} e Cd_AnoExecucao == {TARGET_YEAR}: {rows_before:,}')

    if rows_before == 0:
        raise ValueError(
            'Fallback aplicado, mas nenhum registro encontrado para o ano alvo. '
            f'Ano alvo: {TARGET_YEAR}. '
            'Verifique os par?metros ORCAMENTO_EXECUCAO_ANO e a disponibilidade da base.'
        )

print(f'Modo de leitura utilizado: {source_mode}')
print(f'Fonte efetiva utilizada: {used_source_url}')

df_treated = df_raw[EXPECTED_COLUMNS].copy()
df_treated.head()

In [ ]:
ZERO_TOKENS = {'', '-', '--', '---', 'NA', 'N/A', 'NAN', 'NULL', 'NONE'}

def parse_numeric_column(series: pd.Series) -> tuple[pd.Series, int, int]:
    original = series.fillna('').astype(str).str.strip()
    normalized = original.str.replace('R$', '', regex=False)
    normalized = normalized.str.replace(' ', '', regex=False)
    normalized = normalized.str.replace('\u00A0', '', regex=False)

    has_dot = normalized.str.contains('.', regex=False)
    has_comma = normalized.str.contains(',', regex=False)
    has_both = has_dot & has_comma

    normalized.loc[has_both] = normalized.loc[has_both].str.replace('.', '', regex=False)
    normalized = normalized.str.replace(',', '.', regex=False)

    numeric = pd.to_numeric(normalized, errors='coerce')
    zero_mask = original.str.upper().isin(ZERO_TOKENS)
    invalid_mask = (~zero_mask) & original.ne('') & numeric.isna()

    numeric = numeric.fillna(0.0)
    return numeric, int(invalid_mask.sum()), int(zero_mask.sum())

In [ ]:
conversion_logs = []
for col in VALUE_COLUMNS:
    converted, invalid_count, zero_like_count = parse_numeric_column(df_treated[col])
    df_treated[col] = converted
    conversion_logs.append({
        'coluna': col,
        'valores_invalidos_convertidos_para_zero': invalid_count,
        'valores_zero_ou_vazio_convertidos_para_zero': zero_like_count
    })

df_conversion_logs = pd.DataFrame(conversion_logs)
df_conversion_logs

In [ ]:
rows_after = len(df_treated)
if rows_before != rows_after:
    raise ValueError(f'Quantidade de linhas alterada indevidamente: {rows_before} -> {rows_after}')

if df_treated.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError('A ordem final das colunas n?o corresponde ao esperado.')

print(f'Linhas na sa?da: {rows_after:,}')
print(f'Colunas na sa?da: {len(df_treated.columns)}')
df_treated.dtypes

In [ ]:
if not path.exists(OUTPUT_DIR):
    makedirs(OUTPUT_DIR)

df_treated.to_csv(
    OUTPUT_PATH,
    index=False,
    sep=';',
    decimal=',',
    encoding='latin1'
)

print(f'Arquivo exportado em: {OUTPUT_PATH}')

In [ ]:
df_check = pd.read_csv(OUTPUT_PATH, sep=';', encoding='latin1', dtype=str)
if df_check.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError('Valida??o p?s-escrita falhou: colunas do arquivo exportado n?o conferem.')

print('Valida??o p?s-escrita conclu?da com sucesso.')
print(f'Colunas exportadas: {len(df_check.columns)}')
df_check.head()